# Blog 05 — Meet the Smallest Neural Network

> 🧪 **[Open in Google Colab](https://colab.research.google.com/github/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/notebooks/05-the-smallest-neural-network.ipynb)**

📖 **[Read the matching blog](https://github.com/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/blogs/05-the-smallest-neural-network.md)**

**Core idea:** a neuron is multiply + add + bias. The lab starts with arithmetic, then builds the same operation with NumPy and PyTorch.

## 🧭 Lesson bridge

**Came from:** Blog 04 — matrices transform vectors.

**Today:** turn that idea into a neuron.

**Next:** Blog 06 — activation functions add nonlinearity.


## 1. Learning objectives
By the end you can:
- calculate a neuron by hand;
- explain weights and bias;
- use NumPy `@` for the dot product;
- express a layer as matrix multiplication;
- verify the same mathematics in PyTorch;
- predict the effect of changing one parameter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## 2. Hand calculation first

Take `x = [2, 3, 4]`, `w = [4, 5, 2]`, and `b = 1`.

$$z=w^Tx+b=4(2)+5(3)+2(4)+1=32.$$

Before running the next cell, calculate it yourself.

In [ ]:
x = np.array([2.0, 3.0, 4.0])
w = np.array([4.0, 5.0, 2.0])
b = 1.0

z = w @ x + b
print("z =", z)
assert np.isclose(z, 32.0)


### What does `@` mean?

For two vectors, `w @ x` computes the dot product: multiply matching entries and add them. This is the exact arithmetic used in the neuron equation.

In [ ]:
contributions = w * x
print("individual contributions:", contributions)
print("sum of contributions:", contributions.sum())
print("bias:", b)
print("final z:", contributions.sum() + b)
assert np.isclose(contributions.sum() + b, 32.0)


## 3. Parameter experiment

Change **only one** parameter in each experiment. Predict first.

- Increasing `b` should shift the output by exactly the same amount.
- Increasing a weight changes the output according to its input.
- A negative weight creates a negative contribution.

In [ ]:
baseline = w @ x + b
bias_changed = w @ x + 4.0
first_weight_changed = np.array([6.0, 5.0, 2.0]) @ x + b
negative_weight = np.array([4.0, -5.0, 2.0]) @ x + b

print({"baseline": baseline, "bias=4": bias_changed,
       "w[0]=6": first_weight_changed, "w[1]=-5": negative_weight})
assert np.isclose(bias_changed - baseline, 3.0)
assert np.isclose(first_weight_changed - baseline, 4.0)


## 4. One neuron as a line

For one input, `y = wx + b` is a straight line. Weight controls slope; bias controls vertical shift.

In [ ]:
x_line = np.linspace(-3, 3, 100)
plt.plot(x_line, 2*x_line + 1, label="w=2, b=1")
plt.plot(x_line, 0.5*x_line + 1, label="w=0.5, b=1")
plt.plot(x_line, 2*x_line - 2, label="w=2, b=-2")
plt.xlabel("input x")
plt.ylabel("output z")
plt.title("Weight changes slope; bias shifts the line")
plt.legend()
plt.show()


## 5. Many neurons = one matrix operation

Let

$$W=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix},\quad x=\begin{bmatrix}2\\3\end{bmatrix}.$$

Each row is one neuron's weight vector.

In [ ]:
W = np.array([[1., 2.], [3., 4.], [5., 6.]])
x2 = np.array([2., 3.])
b2 = np.array([0., 0., 0.])
z_layer = W @ x2 + b2
print(z_layer)
assert np.allclose(z_layer, [8., 18., 28.])


### Batch form

Now put examples in rows:

$$X=\begin{bmatrix}1&2\\2&3\\3&4\end{bmatrix},\quad w=\begin{bmatrix}2\\3\end{bmatrix},\quad b=1.$$

All predictions can be calculated as `X @ w + b`.

In [ ]:
X = np.array([[1.,2.], [2.,3.], [3.,4.]])
w_single = np.array([2.,3.])
b_single = 1.
predictions = X @ w_single + b_single
print(predictions)
assert np.allclose(predictions, [9.,14.,19.])


## 6. Parameter counting

A neuron with `d` inputs has `d` weights + one bias = `d + 1` parameters.

A layer with `d` inputs and `m` neurons has `m*d + m = m(d+1)` parameters.

For 4 inputs and 3 neurons: `3*4 + 3 = 15`.

In [ ]:
d, m = 4, 3
parameter_count = m*d + m
print("parameters =", parameter_count)
assert parameter_count == 15


## 7. Visual neuron

The next picture makes each contribution visible. Try changing one weight or input and predict which bar will change.

In [ ]:
labels = ["x1*w1", "x2*w2", "x3*w3", "bias"]
values = [x[0]*w[0], x[1]*w[1], x[2]*w[2], b]
plt.bar(labels, values)
plt.axhline(0)
plt.ylabel("contribution to z")
plt.title("Inside one neuron: contributions are added")
plt.show()


## 8. PyTorch: verify the mathematics

PyTorch's `nn.Linear` implements the same affine operation `xW^T + b` for a batch. We first reproduce the scalar arithmetic manually, then use the layer API.

In [ ]:
import torch

tx = torch.tensor([2., 3., 4.])
tw = torch.tensor([4., 5., 2.])
tb = torch.tensor(1.)
tz = tw @ tx + tb
print("manual PyTorch z =", tz.item())
assert torch.isclose(tz, torch.tensor(32.))


In [ ]:
layer = torch.nn.Linear(3, 1)
with torch.no_grad():
    layer.weight.copy_(tw.reshape(1, 3))
    layer.bias.copy_(tb.reshape(1))

layer_output = layer(tx)
print("nn.Linear output =", layer_output.item())
assert torch.isclose(layer_output.squeeze(), torch.tensor(32.))


## 9. A tiny interactive animation

This animation shows a line changing as the weight changes. It is a visual bridge to later optimization lessons.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

xs = np.linspace(-3, 3, 100)
weights = np.linspace(-1.5, 3.0, 50)
fig, ax = plt.subplots()
ax.set_xlim(-3, 3)
ax.set_ylim(-6, 10)
line, = ax.plot([], [])

def draw(i):
    weight = weights[i]
    line.set_data(xs, weight*xs + 1)
    ax.set_title(f"Neuron: z = {weight:.2f}x + 1")
    return (line,)

ani = FuncAnimation(fig, draw, frames=len(weights), interval=60, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


## 10. Failure mode: affine is not nonlinear

A neuron containing only `w @ x + b` can represent affine relationships. Stacking affine layers without nonlinear activations still produces an affine transformation.

This is the reason Blog 06 matters: we need a nonlinear function between layers to build richer functions.

In [ ]:
# Two affine functions collapse into one affine function.
x_test = np.linspace(-2, 2, 20)
y_two = 3*(2*x_test + 1) - 4
y_one = 6*x_test - 1
assert np.allclose(y_two, y_one)
print("Two affine layers are still one affine function.")


## 11. Challenges

**Beginner:** calculate a neuron for `x=[1,4]`, `w=[2,-3]`, `b=5`.

**Builder:** create a layer with 5 inputs and 4 neurons. Verify its parameter count and output shape.

**Scientist:** vary one weight across 11 values, record the output, and explain why the output changes linearly with that weight.

**Research bridge:** investigate how dense-layer parameter count affects memory and FLOPs. Then ask how parameter sharing in convolution changes the story.

## 🏁 Mastery gate

Do not move on until you can explain `z = w^T x + b` without looking it up, calculate it by hand, reproduce it in NumPy, verify it in PyTorch, and predict what happens when one weight or the bias changes.

**Next:** Blog 06 — Why Does a Neuron Need an Activation Function?